# NumPy Shapes and Matrix Multiplication

**Goal.** Get comfortable with NumPy matrix multiplication by building a linear layer that mixes features. Every axis in that product has a meaning.

**Objective.** For any tensor in that layer, be able to answer two questions:

1. What does each axis mean?
2. What shape must the result have?


Most of a modern language model is matrices. The billions of parameters you hear about are the numbers inside those matrices.

In the literature those same objects are usually called tensors. Tensor is the general word. They differ only in how many axes they have.

- A scalar is a tensor with zero axes.
- A vector is a tensor with one axis.
- A matrix is a tensor with two axes.

This notebook practices matrix multiplication. A matrix product uses two axes. Each row of the first matrix is combined with each column of the second. The two sizes that meet in the middle must be equal, and that shared axis does not appear in the result.

A tensor with more than two axes does not use a different product. The extra axes only count how many matrices you have. In the picture below, `(2, 3, 4)` means two matrices of `3 x 4`. The `2` is not multiplied. Each `3 x 4` matrix is multiplied by the same `4 x 5` matrix, and you get two `3 x 5` results. The result shape is `(2, 3, 5)`.

```text
(2, 3, 4)                    (4, 5)                 (2, 3, 5)

+----------------+                                  +----------------+
|                |           +----------+           |                |
|   matrix 0     |      @    |          |           |   result 0     |
|    3 x 4       |           |  4 x 5   |      =    |    3 x 5       |
|                |           |          |           |                |
+----------------+           +----------+           +----------------+
|                |                ^                 |                |
|   matrix 1     |                |                 |   result 1     |
|    3 x 4       |           this 4 must equal      |    3 x 5       |
|                |           the 4 in each matrix   |                |
+----------------+           on the left. Then      +----------------+
       ^                     that axis is gone.            ^
       |                                                   |
  this 2 is not multiplied.                          the same 2.
  It only means "two matrices."                       It still means two.
```

If the right side is also 3D, it is a stack too. Each matrix on the left meets the matrix in the same place on the right. In the picture below, `(2, 3, 4) @ (2, 4, 5)` is two separate products: matrix 0 with matrix 0, and matrix 1 with matrix 1. The result is still `(2, 3, 5)`.

```text
(2, 3, 4)                 (2, 4, 5)                 (2, 3, 5)

+----------------+        +----------------+        +----------------+
|                |        |                |        |                |
|   matrix 0     |   @    |   matrix 0     |   =    |   result 0     |
|    3 x 4       |        |    4 x 5       |        |    3 x 5       |
|                |        |                |        |                |
+----------------+        +----------------+        +----------------+
|                |        |                |        |                |
|   matrix 1     |   @    |   matrix 1     |   =    |   result 1     |
|    3 x 4       |        |    4 x 5       |        |    3 x 5       |
|                |        |                |        |                |
+----------------+        +----------------+        +----------------+
       ^                         ^                         ^
       |                         |                         |
  two matrices              the partner for          still two results.
  on the left.              each one. The 4          matrix 0 did not
                            must match, then         meet matrix 1.
                            that axis is gone.
```

You may hear this called tensor multiplication. It is this same matrix product, repeated once for each matrix in the stack.

Here is the same idea with four axes. Take `(2, 3, 4, 5) @ (5, 6)`.

`(2, 3, 4, 5)` means 6 matrices, because `2 × 3 = 6`. Each matrix is `4 × 5`. They are not a loose pile. They are arranged as 2 groups of 3. The right side is one matrix, `5 × 6`. Each of the 6 matrices is multiplied by that same matrix. `4 × 5` times `5 × 6` is `4 × 6`. You still have 2 groups of 3, so the result is `(2, 3, 4, 6)`.

That is the rule. Only the last two axes are multiplied. Every axis in front of them only says how the matrices are arranged. If the right side is one matrix, every matrix on the left meets that same matrix. If the right side is arranged too, each matrix meets only the matrix in the same place in that arrangement.

- `(2, 3, 4, 5) @ (2, 3, 5, 6)` is valid. Both sides are 2 groups of 3, and the inner sizes are both 5. The matrix in each place meets the matrix in that same place. The result is `(2, 3, 4, 6)`.
- `(2, 3, 4, 5) @ (3, 5, 6)` is valid. The right side has one group of 3 matrices, each `5 × 6`. That group of 3 lines up with the 3 on the left, and it is reused for both of the left side's groups. The inner sizes are both 5. The result is `(2, 3, 4, 6)`.
- `(2, 3, 4, 5) @ (2, 3, 7, 6)` is invalid. The matrices would be `4 × 5` and `7 × 6`. The inner sizes are 5 and 7, so they cannot be multiplied.
- `(2, 3, 4, 5) @ (9, 3, 5, 6)` is invalid. The left matrices are arranged as 2 by 3, and the right matrices are arranged as 9 by 3. A matrix needs a partner in the same place, and 2 is not the same count as 9.

**Remember.** Read the shape from the right. The last two sizes are the matrix. The sizes in front only say how many of those matrices there are, and how they are arranged.

`(2, 3, 4, 5)`: six `4 × 5` matrices, arranged as 2 by 3.

Once the tensors are large, you will rely on checking these shapes instead of individual numbers. A lot of the later GPU work is about making these multiplications fast.

The next cell writes these objects as ordinary Python lists. The nesting is the shape.


In [ ]:
# Scalar. Shape (). No axes, so 0D.
# D means dimensions: the number of axes in the shape.
x = 5

# Vector. Shape (3,). One axis, so 1D.
y = [1, 2, 3]

# Matrix. Shape (2, 3): 2 rows and 3 columns. Two axes, so a 2D matrix.
z = [
    [1, 2, 3],
    [4, 5, 6],
]

# Tensor. Shape (2, 2, 3): two matrices, each 2 by 3. Three axes, so 3D.
w = [
        [
            [1, 2, 3],
            [4, 5, 6],
        ],
        [
            [7, 8, 9],
            [10, 11, 12],
        ],
    ]

# 4D means four axes. A common shape is (batch, heads, tokens, features).
# 5D means five axes. A common shape is (batch, layers, heads, tokens, features).
# You will not write these out number by number. They show up when smaller tensors are stacked.


**What the D means.** D stands for dimension, which here means an axis. A 3D tensor has three axes. It is not a 3D picture. In NumPy, the number of axes is `ndim`.

You will not type 4D or 5D tensors out by hand. They appear when smaller tensors are stacked. A batch of token vectors is already 3D: batch, tokens, features. Split each vector across attention heads and you have 4D. You read the shape. You do not picture every cell.

Let us now understand the intuition behind a matrix and develop a mental model.

Put a point on a page with ordinary axes. Its coordinates `(x, y)` say how far it sits along the x-axis and the y-axis. The point `(2, 3)` is 2 units to the right of the origin and 3 units up.

![P is 2 steps along x, then 3 steps along y](figures/point.png)

`P` is made from the two arrows: `2 × [1, 0] + 3 × [0, 1]`. `[1, 0]` is one step along the x-axis. `[0, 1]` is one step along the y-axis.

A matrix moves those arrows. It records where the old x-axis lands and where the old y-axis lands. `P` moves with them, and it is still "2 along the new x-axis, plus 3 along the new y-axis." The coordinates change because the arrows they were counted on have moved.

A stretch changes distance. If the x-axis lands at `(2, 0)` and the y-axis stays at `(0, 1)`, the point `(1, 0)` becomes `(2, 0)`. It is the same direction, twice as far. A zoom on a map is this.

![Before, P sits at the tip of the x-axis. After x is stretched by 2, that axis and P both end at (2, 0). The y-axis stays put. The gray arrow is where the x-axis used to be.](figures/stretch.png)

`P` moves from `(1, 0)` to `(2, 0)`. The y-axis stays where it is.

A turn changes direction. A quarter turn to the left sends the x-axis to `(0, 1)` and the y-axis to `(-1, 0)`. The point that was on the x-axis now sits on the y-axis. Turning a photo does this to every corner.

![Before, P sits at the tip of the x-axis. After a quarter turn left, the x-axis points up and P moves with it. The y-axis points left. Gray arrows show where the axes used to be.](figures/quarter_turn.png)

`P` moves from `(1, 0)` to `(0, 1)`. The x-axis now points up. The y-axis now points left.

Multiplying two matrices is one move followed by another. The product is a single matrix that does both. Turn and then stretch is not the same as stretch and then turn, so the order matters.

Next, the same objects are stored in NumPy, and we start operating on them.

In [11]:
# basic introduction to numpy
import numpy as np

np.set_printoptions(suppress=True)

def describe(name, array):
    print(f'{name}: shape={array.shape}, ndim={array.ndim}, dtype={array.dtype}')
    print(array)
    print("\n")


## 1. The same objects in NumPy

The lists above are ordinary Python. NumPy stores the same idea in an array.

- `ndim` is the D from above. It counts the axes.
- `shape` is the size of each axis.
- Array is the container. The word itself does not say how many axes there are.

A vector shaped `(2,)` is not a column shaped `(2, 1)`. That extra axis changes the multiplication later.


In [12]:
scalar = np.array(3.0)            # 0D
vector = np.array([2.0, -1.0])   # 1D
matrix = np.array([[1.0, 0.0],
                   [0.0, 1.0]])  # 2D tensor, a matrix
# New axis in front. Two matrices, each still 2 x 2. Shape (2, 2, 2).
tensor = np.stack([matrix, matrix])

describe('scalar', scalar)
describe('vector', vector)
describe('matrix', matrix)
describe('tensor', tensor)

assert scalar.shape == () and scalar.ndim == 0
assert vector.shape == (2,) and vector.ndim == 1
assert matrix.shape == (2, 2) and matrix.ndim == 2
assert tensor.shape == (2, 2, 2) and tensor.ndim == 3
assert all(isinstance(x, np.ndarray) for x in (scalar, vector, matrix, tensor))


scalar: shape=(), ndim=0, dtype=float64
3.0


vector: shape=(2,), ndim=1, dtype=float64
[ 2. -1.]


matrix: shape=(2, 2), ndim=2, dtype=float64
[[1. 0.]
 [0. 1.]]


tensor: shape=(2, 2, 2), ndim=3, dtype=float64
[[[1. 0.]
  [0. 1.]]

 [[1. 0.]
  [0. 1.]]]




`np.stack` adds a new axis and keeps each matrix as it was. `tensor[0]` is the first `2 × 2` matrix. `tensor[1]` is the second.

`vstack` and `hstack` glues the matrices together into one bigger matrix.

- `vstack` puts one matrix under the other. Two `2 × 2` matrices become shape `(4, 2)`.
- `hstack` puts one matrix beside the other. Two `2 × 2` matrices become shape `(2, 4)`.


In [13]:
under = np.vstack([matrix, matrix])
beside = np.hstack([matrix, matrix])

describe('under', under)
describe('beside', beside)

assert under.shape == (4, 2)
assert beside.shape == (2, 4)
assert np.array_equal(tensor[0], matrix)
assert np.array_equal(tensor[1], matrix)


under: shape=(4, 2), ndim=2, dtype=float64
[[1. 0.]
 [0. 1.]
 [1. 0.]
 [0. 1.]]


beside: shape=(2, 4), ndim=2, dtype=float64
[[1. 0. 1. 0.]
 [0. 1. 0. 1.]]




## 3. Dot product: one output number

A dot product lines up two vectors, multiplies each pair of numbers, and adds those products. Two vectors of the same length become one number. That number is a scalar, and its shape is `()`.


In [20]:
def dot_product(x, w):
    """Multiply paired entries of two vectors and add the products.

    Each entry of ``x`` is multiplied by the entry of ``w`` in the same
    position. Those products are then added. The vectors must have the same
    length. If one is longer, the extra entries have no partner, so the
    product is not defined.

    Args:
        x (np.ndarray): First vector.
        w (np.ndarray): Second vector. Must have the same length as ``x``.

    Returns:
        float: The sum of the paired products.

    Raises:
        ValueError: If ``len(x)`` and ``len(w)`` differ.
    """
    # Unequal lengths have no paired entries, so the product is not defined.
    if len(x) != len(w):
        raise ValueError("Vectors must be of the same length")
    product = 0.0
    for i in range(len(x)):
        product += x[i] * w[i]
    return product

def np_dot_product(x, w):
    """Compute the dot product with NumPy.

    ``@`` applies the same rule. Different lengths raise ``ValueError``.

    Args:
        x (np.ndarray): First vector.
        w (np.ndarray): Second vector of the same length.

    Returns:
        np.ndarray: One number, with shape ``()``.

    Raises:
        ValueError: If the two vectors have different lengths.
    """
    return x @ w

x = np.array([2.0, -1.0])
w = np.array([0.25, 0.75])
z = np.array([0.25, 0.75, 0.5])  # one entry longer than x

print('same length, manual:', dot_product(x, w))
print('same length, NumPy :', np_dot_product(x, w))
assert np.isclose(dot_product(x, w), np_dot_product(x, w))
assert np.ndim(np_dot_product(x, w)) == 0

def show_mismatch(name, function):
    """Print the ValueError raised when the vectors have different lengths."""
    try:
        function(x, z)
    except ValueError as error:
        print(f'{name}: {type(error).__name__}: {error}')
        return
    raise AssertionError(f'{name} should reject unequal lengths')

show_mismatch('manual', dot_product)
show_mismatch('NumPy ', np_dot_product)


same length, manual: -0.25
same length, NumPy : -0.25
manual: ValueError: Vectors must be of the same length
NumPy : ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 3 is different from 2)


## 4. Matrix multiplication: many dot products

A dot product takes one vector from the left and one vector from the right, and returns one number. A matrix multiplication is that same dot product, taken between each row of the left matrix and each column of the right matrix.

Consider `(3, 2) @ (2, 4)`.

The left matrix has 3 rows, and each row has length 2. The right matrix has 4 columns, and each column has length 2. Pair each row with each column and you get `3 × 4` numbers, so the result shape is `(3, 4)`.

The shared length, 2, has to match. That is the same rule as the dot product: an entry with no partner makes the product undefined. That shared length is used up and does not appear in the result. The 3 and the 4 stay, because they count the rows and the columns.

```text
(3, 2) @ (2, 4) -> (3, 4)
```


In [34]:
def matrix_multiplication(mat1, mat2):
    """Dot each row of mat1 with each column of mat2.

    Args:
        mat1 (np.ndarray): Left matrix, shape (rows, shared).
        mat2 (np.ndarray): Right matrix, shape (shared, columns).

    Returns:
        np.ndarray: Shape (rows, columns). Entry ``[i, j]`` is the dot product
            of row ``i`` of ``mat1`` and column ``j`` of ``mat2``.

    Raises:
        ValueError: If the columns of ``mat1`` and the rows of ``mat2`` differ.
    """
    if mat1.shape[1] != mat2.shape[0]:
        raise ValueError("Number of columns in mat1 must match number of rows in mat2")

    rows = []
    for row in mat1:
        rows.append([dot_product(row, column) for column in mat2.T])
    # Equal-length inner lists become a matrix. Shape is inferred from the nesting.
    return np.array(rows)

def np_matrix_multiplication(mat1, mat2):
    """Compute the same matrix product with NumPy.

    Args:
        mat1 (np.ndarray): Left matrix, shape (rows, shared).
        mat2 (np.ndarray): Right matrix, shape (shared, columns).

    Returns:
        np.ndarray: Shape (rows, columns).

    Raises:
        ValueError: If the shared lengths differ.
    """
    return mat1 @ mat2

# Three rows, each a vector of length 2.
tokens = np.array([
    [2.0, -1.0],
    [0.5, 3.0],
    [-2.0, 1.0],
])  # (3, 2)

# Four columns, each a vector of length 2.
weights = np.array([
    [1.0, 0.0, -1.0, 2.0],
    [0.0, 1.0,  1.0, -1.0],
])  # (2, 4)

manual = matrix_multiplication(tokens, weights)
numpy_result = np_matrix_multiplication(tokens, weights)

describe('tokens', tokens)
describe('weights', weights)
describe('manual', manual)
describe('NumPy', numpy_result)

assert manual.shape == (3, 4)
assert np.allclose(manual, numpy_result)
# First row [2, -1] dotted with each column: [1, 0], [0, 1], [-1, 1], [2, -1].
assert np.allclose(manual[0], [2.0, -1.0, -3.0, 5.0])
assert np.isclose(manual[0, 0], dot_product(tokens[0], weights[:, 0]))


tokens: shape=(3, 2), ndim=2, dtype=float64
[[ 2.  -1. ]
 [ 0.5  3. ]
 [-2.   1. ]]


weights: shape=(2, 4), ndim=2, dtype=float64
[[ 1.  0. -1.  2.]
 [ 0.  1.  1. -1.]]


manual: shape=(3, 4), ndim=2, dtype=float64
[[ 2.  -1.  -3.   5. ]
 [ 0.5  3.   2.5 -2. ]
 [-2.   1.   3.  -5. ]]


NumPy: shape=(3, 4), ndim=2, dtype=float64
[[ 2.  -1.  -3.   5. ]
 [ 0.5  3.   2.5 -2. ]
 [-2.   1.   3.  -5. ]]




## 5. `*` versus `@`

`*` operates element by element. `@` performs matrix multiplication. Confusing them is one of the most common sources of silent model bugs.


In [36]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print('a * b (elementwise):\n', a * b)
print("\n")
print('a @ b (matrix product):\n', a @ b)

assert np.array_equal(a * b, [[5, 12], [21, 32]])
assert np.array_equal(a @ b, [[19, 22], [43, 50]])


a * b (elementwise):
 [[ 5 12]
 [21 32]]


a @ b (matrix product):
 [[19 22]
 [43 50]]


## 6. Several matrices, the same weights

The last cell multiplied one matrix by one matrix:

```text
(3, 2) @ (2, 4) -> (3, 4)
```

Stack a second `3 × 2` matrix with `tokens` and the shape becomes `(2, 3, 2)`. Read that from the right. The last two sizes are still the matrix, `3 × 2`. The `2` in front only says there are two of them.

Multiply the stack by the same weights.

```text
(2, 3, 2) @ (2, 4) -> (2, 3, 4)
```

Each `3 × 2` is dotted with the columns of that same `2 × 4`. The shared length 2 is used up, just as in the dot product. The leading 2 stays, so the result is two matrices of shape `(3, 4)`. The first is the product we already computed. The second is the other matrix, times the same weights.

This is the picture from the start of the notebook. Only the last two axes are multiplied. The axis in front is the arrangement, and it is still there afterwards.

If the weights are a stack too, each left matrix meets the matrix in the same place. `(2, 3, 2) @ (2, 2, 4)` means two matrices on the left and two on the right. Pair them by place. The result is still `(2, 3, 4)`.

`(2, 3, 2) @ (3, 2, 4)` is undefined. The `3` at the front of the right shape means three matrices. The left shape has two. Each matrix needs a partner in the same place, and two is not the same count as three.


In [38]:
# A second 3 by 2 matrix, the same shape as tokens.
more = np.array([
    [1.0, 1.0],
    [0.0, 2.0],
    [4.0, -1.0],
])  # (3, 2)

# Two matrices, each 3 by 2. The new axis is in front.
batch = np.stack([tokens, more])  # (2, 3, 2)

# The same weights for both. The leading 2 stays.
batched = batch @ weights  # (2, 3, 4)

describe('batch', batch)
describe('batched', batched)

assert batched.shape == (2, 3, 4)
assert np.allclose(batched[0], manual)
# First row of the second matrix, [1, 1], dotted with each column of weights.
assert np.allclose(batched[1, 0], [1.0, 1.0, 0.0, 1.0])
assert np.isclose(batched[1, 0, 0], dot_product(more[0], weights[:, 0]))

# The right side is a stack as well. Matrix 0 meets matrix 0. Matrix 1 meets matrix 1.
other_weights = np.array([
    [1.0, 0.0, 0.0, 1.0],
    [0.0, 1.0, 1.0, 0.0],
])  # (2, 4)
paired = batch @ np.stack([weights, other_weights])  # right side is (2, 2, 4)

describe('paired', paired)
assert paired.shape == (2, 3, 4)
assert np.allclose(paired[0], tokens @ weights)
assert np.allclose(paired[1], more @ other_weights)


batch: shape=(2, 3, 2), ndim=3, dtype=float64
[[[ 2.  -1. ]
  [ 0.5  3. ]
  [-2.   1. ]]

 [[ 1.   1. ]
  [ 0.   2. ]
  [ 4.  -1. ]]]


batched: shape=(2, 3, 4), ndim=3, dtype=float64
[[[ 2.  -1.  -3.   5. ]
  [ 0.5  3.   2.5 -2. ]
  [-2.   1.   3.  -5. ]]

 [[ 1.   1.   0.   1. ]
  [ 0.   2.   2.  -2. ]
  [ 4.  -1.  -5.   9. ]]]


paired: shape=(2, 3, 4), ndim=3, dtype=float64
[[[ 2.  -1.  -3.   5. ]
  [ 0.5  3.   2.5 -2. ]
  [-2.   1.   3.  -5. ]]

 [[ 1.   1.   1.   1. ]
  [ 0.   2.   2.   0. ]
  [ 4.  -1.  -1.   4. ]]]




## 7. Add one vector to every row

`batched` has shape `(2, 3, 4)`. Every row has length 4. A vector of that same length can be added to every row at once.

```text
(2, 3, 4) + (4,) -> (2, 3, 4)
```

NumPy lines the shapes up from the right. The `4` meets the `4`. The vector has no axis in front of that, so those four numbers are reused for every row of both matrices. The shape stays `(2, 3, 4)`.

The first row we already know is `[2, -1, -3, 5]`. Add `[0.1, 0.2, 0.3, 0.4]` and that row becomes `[2.1, -0.8, -2.7, 5.4]`. The first row of the second matrix, `[1, 1, 0, 1]`, becomes `[1.1, 1.2, 0.3, 1.4]`. Same four numbers, every row.

This is the `+ b` in `x @ W + b`. The product builds each row. The vector then shifts every row by the same amount. That vector is the bias.

A vector of length 3 does not line up with the 4, so the add is undefined. The match is checked from the right. The `3` in the middle of `(2, 3, 4)` is not the partner.


In [39]:
bias = np.array([0.1, 0.2, 0.3, 0.4])  # (4,)
with_bias = batched + bias  # (2, 3, 4)

describe('bias', bias)
describe('with_bias', with_bias)

assert with_bias.shape == (2, 3, 4)
assert np.allclose(with_bias[0, 0], [2.1, -0.8, -2.7, 5.4])
assert np.allclose(with_bias[1, 0], [1.1, 1.2, 0.3, 1.4])
assert np.allclose(with_bias[0, 0], batched[0, 0] + bias)

try:
    batched + np.array([0.1, 0.2, 0.3])  # (3,) against rows of length 4
except ValueError as error:
    print(f'length 3: {type(error).__name__}: {error}')
else:
    raise AssertionError('a vector of length 3 should not add to rows of length 4')


bias: shape=(4,), ndim=1, dtype=float64
[0.1 0.2 0.3 0.4]


with_bias: shape=(2, 3, 4), ndim=3, dtype=float64
[[[ 2.1 -0.8 -2.7  5.4]
  [ 0.6  3.2  2.8 -1.6]
  [-1.9  1.2  3.3 -4.6]]

 [[ 1.1  1.2  0.3  1.4]
  [ 0.1  2.2  2.3 -1.6]
  [ 4.1 -0.8 -4.7  9.4]]]


length 3: ValueError: operands could not be broadcast together with shapes (2,3,4) (3,) 


## 8. One expression: `x @ W + b`

The product and the bias are one line.

```text
(3, 2) @ (2, 4) + (4,) -> (3, 4)
```

`x` holds the rows. `W` holds the columns those rows are dotted with. `b` is added to every row. The first row is `[2, -1]`. Dot it with the columns of `W` and you get `[2, -1, -3, 5]`. Add `b` and that row is `[2.1, -0.8, -2.7, 5.4]`.

The stack uses the same line. The axis in front stays, and `b` is still one vector of length 4.

```text
(2, 3, 2) @ (2, 4) + (4,) -> (2, 3, 4)
```


In [40]:
x = np.array([
    [2.0, -1.0],
    [0.5, 3.0],
    [-2.0, 1.0],
])  # (3, 2)

W = np.array([
    [1.0, 0.0, -1.0, 2.0],
    [0.0, 1.0,  1.0, -1.0],
])  # (2, 4)

b = np.array([0.1, 0.2, 0.3, 0.4])  # (4,)

y = x @ W + b  # (3, 4)

describe('y', y)
assert y.shape == (3, 4)
assert np.allclose(y[0], [2.1, -0.8, -2.7, 5.4])

# Two matrices, each 3 by 2. The new axis is in front.
x_batch = np.stack([
    x,
    [
        [1.0, 1.0],
        [0.0, 2.0],
        [4.0, -1.0],
    ],
])  # (2, 3, 2)

# Same expression. The leading 2 stays, and b is added to every row.
y_batch = x_batch @ W + b  # (2, 3, 4)
assert y_batch.shape == (2, 3, 4)
assert np.allclose(y_batch[0], y)


y: shape=(3, 4), ndim=2, dtype=float64
[[ 2.1 -0.8 -2.7  5.4]
 [ 0.6  3.2  2.8 -1.6]
 [-1.9  1.2  3.3 -4.6]]




## 9. Checkpoint

Answer these from the shapes. Do not run them.

1. `(2, 3, 2) @ (2, 4)` gave `(2, 3, 4)`. What is `(8, 128, 768) @ (768, 3072)`?
2. What is `(8, 128, 3072) + (3072,)`?
3. What happens if the `@` in question 1 is written as `*`?
4. `(2, 3, 2) @ (2, 2, 4)` gave `(2, 3, 4)`, each left matrix meeting the right matrix in the same place. What is `(8, 128, 768) @ (8, 768, 3072)`?
5. What is `(8, 128, 768) @ (4, 768, 3072)`?

<details>
<summary>Answers</summary>

1. `(8, 128, 3072)`. The last two sizes are the matrix. `768` meets `768` and is used up. `128` rows stay, and `3072` columns stay. The leading `8` is the arrangement, so it stays too: eight matrices, each `128 × 3072`.

2. `(8, 128, 3072)`. The shapes line up from the right, and `3072` matches `3072`. The same vector is added to every row. The `8` and the `128` are filled in by reusing that vector, so the shape stays the same.

3. `*` multiplies entries that sit in the same place. `(8, 128, 768)` and `(768, 3072)` do not line up that way, so NumPy raises `ValueError`. The `@` form is the product from section 4, repeated once for each of the eight matrices.

4. `(8, 128, 3072)`. The last two sizes are again the matrix, and `768` is used up. The leading `8` on the left meets the leading `8` on the right, so there are eight products: matrix 0 with matrix 0, matrix 1 with matrix 1, and so on. That `8` stays in the result.

5. Undefined. The left side holds eight matrices. The `4` at the front of the right side means four matrices. Each matrix needs a partner in the same place, and eight is not the same count as four. The matching `768` in the middle does not fix that.

</details>


## What this establishes

Each question from the previous section can be represented as `x @ W + b`.
Read `(8, 128, 768) @ (768, 3072) + (3072,)` the same way you read `(3, 2) @ (2, 4) + (4,)`. The last two sizes are the matrices. Each of the 128 rows is dotted with the columns of `W`, and the shared 768 is used up. The 8 in front means eight of those matrices, and it stays. `b` has one number for each of the 3072 result columns, and that vector is added to every row. The result is `(8, 128, 3072)`.

That result holds `8 × 128 × 3072` numbers, about three million entries. For the small layers you could look at a row and see the numbers. You cannot do that here. Only check is the shape itself: `(8, 128, 3072)`.

A transformer repeats this layer. Before each product, the numbers have to sit on the right axes.

- Indexing picks out one row, or one matrix from a stack. `y[0]` in the last code cell is indexing.
- Reshape splits one axis into two. That is how one vector becomes several attention heads, and a 3D tensor becomes 4D.
- Transpose swaps two axes, so the product meets the pair of sizes you meant.
- Broadcasting is the bias add. A shorter shape lines up from the right and is reused, as `(4,)` was added to every row of `(3, 4)`.

You need these four to build attention. They arrange `x`, `W`, and `b`. The product is still the one you built here.


## 10. Indexing: pick one part

`stack` has shape `(2, 3, 2)`. Two matrices, each `3 × 2`.

An index removes the axis you name. A slice keeps that axis.

- `stack[0]` is the first matrix, shape `(3, 2)`. The leading 2 is gone.
- `stack[0, 0]` is the first row of that matrix, shape `(2,)`.
- `stack[0, 0, 1]` is one entry, the number `-1`. Its shape is `()`.
- `stack[:, 0]` keeps the leading axis and takes row 0 of each matrix. Shape `(2, 2)`: `[2, -1]` from the first matrix and `[1, 1]` from the second.

The colon means every position on that axis.


In [41]:
stack = np.array([
    [
        [2.0, -1.0],
        [0.5, 3.0],
        [-2.0, 1.0],
    ],
    [
        [1.0, 1.0],
        [0.0, 2.0],
        [4.0, -1.0],
    ],
])  # (2, 3, 2)

first = stack[0]            # (3, 2)
row = stack[0, 0]           # (2,)
entry = stack[0, 0, 1]      # ()
each_first_row = stack[:, 0]  # (2, 2)

describe('stack', stack)
describe('first', first)
describe('row', row)
describe('each_first_row', each_first_row)

assert first.shape == (3, 2)
assert row.shape == (2,)
assert np.ndim(entry) == 0
assert entry == -1.0
assert each_first_row.shape == (2, 2)
assert np.allclose(each_first_row, [[2.0, -1.0], [1.0, 1.0]])


stack: shape=(2, 3, 2), ndim=3, dtype=float64
[[[ 2.  -1. ]
  [ 0.5  3. ]
  [-2.   1. ]]

 [[ 1.   1. ]
  [ 0.   2. ]
  [ 4.  -1. ]]]


first: shape=(3, 2), ndim=2, dtype=float64
[[ 2.  -1. ]
 [ 0.5  3. ]
 [-2.   1. ]]


row: shape=(2,), ndim=1, dtype=float64
[ 2. -1.]


each_first_row: shape=(2, 2), ndim=2, dtype=float64
[[ 2. -1.]
 [ 1.  1.]]




## 11. Reshape: regroup the same numbers

`reshape` changes the axes. It does not change the numbers, or the order they are stored in.

A row of 6 entries can be read as 2 groups of 3.

```text
(6,) -> (2, 3)
```

`6` equals `2 × 3`, so this is allowed. `reshape(2, 4)` is not: `2 × 4` is 8, and the row only has 6 numbers.

The same split turns a length into heads. Two rows of length 4 become `(2, 2, 2)`: 2 rows, 2 heads, 2 numbers in each head.

In [46]:
row = np.array([1, 2, 3, 4, 5, 6])  # (6,)
groups = row.reshape(2, 3)  # (2, 3)

describe('groups', groups)
assert groups.shape == (2, 3)
assert np.array_equal(groups, [[1, 2, 3], [4, 5, 6]])

try:
    row.reshape(2, 4)  # 2 * 4 = 8, but the row has 6 numbers
except ValueError as error:
    print(f'reshape(2, 4): {type(error).__name__}: {error}')
    print("\n")

features = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
])  # (2, 4)

split = features.reshape(2, 2, 2)  # (rows, heads, numbers)
describe('split', split)
assert split.shape == (2, 2, 2)
assert np.array_equal(split[0, 0], [1, 2])
assert np.array_equal(split[0, 1], [3, 4])


groups: shape=(2, 3), ndim=2, dtype=int64
[[1 2 3]
 [4 5 6]]


reshape(2, 4): ValueError: cannot reshape array of size 6 into shape (2,4)


split: shape=(2, 2, 2), ndim=3, dtype=int64
[[[1 2]
  [3 4]]

 [[5 6]
  [7 8]]]




## 12. Transpose: swap axes

A matrix of shape `(2, 3)` transposed is `(3, 2)`. The first row was `[1, 2, 3]`. After the swap, those three numbers are the first column.

With more than two axes, you name the new order. `split` from the previous cell is `(2, 2, 2)`, in the order rows, heads, numbers. `transpose(1, 0, 2)` reads that as "axis 1, then axis 0, then axis 2," so the order becomes heads, rows, numbers.

Head 0 was `[1, 2]` in row 0 and `[5, 6]` in row 1. After the swap, head 0 is one matrix:

```text
[[1, 2],
 [5, 6]]
```

`@` multiplies the last two axes. With heads in front, each head is its own matrix. Leave the heads inside each row, and the same `@` instead multiplies the two heads of one row. Both products have shape `(2, 2, 2)`. The numbers are different, because different axes are the matrix.


In [ ]:
matrix = np.array([
    [1, 2, 3],
    [4, 5, 6],
])  # (2, 3)

swapped = matrix.T  # (3, 2)
describe('swapped', swapped)
assert swapped.shape == (3, 2)
assert np.array_equal(swapped[:, 0], [1, 2, 3])

features = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
])  # (2, 4)
split = features.reshape(2, 2, 2)  # (rows, heads, numbers)

by_head = split.transpose(1, 0, 2)  # (heads, rows, numbers)
describe('by_head', by_head)
assert by_head.shape == (2, 2, 2)
assert np.array_equal(by_head[0], [[1, 2], [5, 6]])
assert np.array_equal(by_head[1], [[3, 4], [7, 8]])

W = np.array([
    [1.0, 1.0],
    [0.0, 1.0],
])  # (2, 2)

# Heads are the matrices. Head 0 is [[1, 2], [5, 6]].
per_head = by_head @ W  # (2, 2, 2)
# Heads stay inside each row. Row 0 is [[1, 2], [3, 4]].
per_row = split @ W  # (2, 2, 2)

describe('per_head', per_head)
describe('per_row', per_row)
assert per_head.shape == (2, 2, 2)
assert np.array_equal(per_head[0], [[1.0, 3.0], [5.0, 11.0]])
assert np.array_equal(per_row[0], [[1.0, 3.0], [3.0, 7.0]])


## 13. Broadcasting: reuse a smaller shape

Start with two rows:

```text
[[1, 2, 3],
 [4, 5, 6]]
```

Add `[0.1, 0.2, 0.3]`. That vector has one number per column, and it has no second row of its own. The same three numbers are added to both rows:

```text
[[1.1, 2.2, 3.3],
 [4.1, 5.2, 6.3]]
```

`1 + 0.1` and `4 + 0.1`. `2 + 0.2` and `5 + 0.2`. `3 + 0.3` and `6 + 0.3`.

Add a column instead. One number for the first row, one number for the second:

```text
[[10],
 [20]]
```

`10` is added to every entry of the first row. `20` is added to every entry of the second:

```text
[[11, 12, 13],
 [24, 25, 26]]
```

`1 + 10`, `2 + 10`, `3 + 10`. Then `4 + 20`, `5 + 20`, `6 + 20`. The column is one entry wide, and each row is three entries wide, so that one entry is copied across the row.

`[0.1, 0.2]` cannot be added to these rows. Each row ends in 3 numbers, and the vector has 2. The sizes are different, and neither size is 1, so there is nothing to copy.

Put the shapes in a column, lined up on the right. A size fits when it equals the other size, or when it is 1. A blank on the left counts as 1, and a 1 is copied out to the other size.

```text
(2, 3)
   (3,)      3 fits 3. The blank is copied down, one copy per row.

(2, 3)
(2, 1)       1 is copied across the 3. 2 fits 2.

(2, 3)
   (2,)      2 against 3. Nothing to copy.
```


In [47]:
rows = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
])  # (2, 3)

per_column = np.array([0.1, 0.2, 0.3])  # (3,) copied down both rows
per_row = np.array([
    [10.0],
    [20.0],
])  # (2, 1) copied across each row

added_columns = rows + per_column  # (2, 3)
added_rows = rows + per_row        # (2, 3)

describe('added_columns', added_columns)
describe('added_rows', added_rows)
assert np.allclose(added_columns, [[1.1, 2.2, 3.3], [4.1, 5.2, 6.3]])
assert np.allclose(added_rows, [[11.0, 12.0, 13.0], [24.0, 25.0, 26.0]])

try:
    rows + np.array([0.1, 0.2])  # (2,) against a last axis of 3
except ValueError as error:
    print(f'length 2: {type(error).__name__}: {error}')
else:
    raise AssertionError('a vector of length 2 should not add to rows of length 3')


added_columns: shape=(2, 3), ndim=2, dtype=float64
[[1.1 2.2 3.3]
 [4.1 5.2 6.3]]


added_rows: shape=(2, 3), ndim=2, dtype=float64
[[11. 12. 13.]
 [24. 25. 26.]]


length 2: ValueError: operands could not be broadcast together with shapes (2,3) (2,) 
mask: shape=(3, 3), ndim=2, dtype=float64
[[ 0. -1. -1.]
 [ 0.  0. -1.]
 [ 0.  0.  0.]]


masked[0, 1]: shape=(3, 3), ndim=2, dtype=float64
[[ 0. -1. -1.]
 [ 0.  0. -1.]
 [ 0.  0.  0.]]


